In [1]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

In [2]:
df = pd.read_csv("../../results/invivo_base.csv", index_col=0)
df.baseline.unique(), df.type.unique(), df.groupby(["baseline", "type"]).size()

(array(['Cellina-base', 'mean', 'Cellina-base-random'], dtype=object),
 array(['full', 'KO-specific'], dtype=object),
 baseline             type       
 Cellina-base         KO-specific    9
                      full           9
 Cellina-base-random  KO-specific    9
                      full           9
 mean                 KO-specific    9
                      full           9
 dtype: int64)

## Statistical significance testing

Cellina-base vs. Cellina-base-random, for the "full" and "KO-specific" in-vivo runs separately, paired by KO — 9 pairs per run type.
One-sided paired Wilcoxon signed-rank test per metric (direction chosen so the alternative is "Cellina-base is better"), with Holm-Bonferroni correction across all 8 tests (2 run types × 4 metrics).

Note: with only 9 paired KOs per run type, statistical power is limited — interpret p-values with that in mind.

In [3]:
metrics = ["Pearson", "Precision", "E-distance", "RMSE_LFC"]

# Whether a higher value of the metric means a better prediction
higher_is_better = {
    "Pearson": True,
    "Precision": True,
    "E-distance": False,
    "RMSE_LFC": False,
}

def paired_values(df, run_type, model_a, model_b, metric):
    """Align per-KO scores for two models on one metric, within a given run type."""
    sub = df[df.type == run_type]
    a = sub[sub.baseline == model_a].set_index("KO")[metric]
    b = sub[sub.baseline == model_b].set_index("KO")[metric]
    common = a.index.intersection(b.index)
    return a.loc[common], b.loc[common]

def win_rate(a, b, higher_better):
    wins = (a.values > b.values) if higher_better else (a.values < b.values)
    return wins.sum() / len(a)

In [4]:
model_a, model_b = "Cellina-base", "Cellina-base-random"
run_types = ["full", "KO-specific"]

rows = []
for run_type in run_types:
    for metric in metrics:
        a, b = paired_values(df, run_type, model_a, model_b, metric)
        alternative = "greater" if higher_is_better[metric] else "less"
        stat, p = wilcoxon(a, b, alternative=alternative)
        rows.append({
            "type": run_type,
            "metric": metric,
            "n_pairs": len(a),
            "mean_a": a.mean(),
            "mean_b": b.mean(),
            "cellina_base_win_rate": win_rate(a, b, higher_is_better[metric]),
            "statistic": stat,
            "p_value": p,
        })

stats_df = pd.DataFrame(rows)

# Holm-Bonferroni correction across the full family of tests (2 run types x 4 metrics)
reject, p_adj, _, _ = multipletests(stats_df["p_value"], method="holm")
stats_df["p_adj"] = p_adj
stats_df["significant"] = reject

stats_df

,type,metric,n_pairs,mean_a,mean_b,cellina_base_win_rate,statistic,p_value,p_adj,significant
0,full,Pearson,9,0.583990,0.216109,0.777778,42.0,0.009766,0.078125,False
1,full,Precision,9,0.788889,0.584444,0.888889,41.0,0.013672,0.095703,False
2,full,E-distance,9,4.633645,4.454575,0.000000,45.0,1.000000,1.000000,False
3,full,RMSE_LFC,9,0.463146,0.546406,0.777778,5.0,0.019531,0.117188,False
4,KO-specific,Pearson,9,0.321283,-0.036876,0.666667,37.0,0.048828,0.195312,False
5,KO-specific,Precision,9,0.666667,0.464444,0.666667,39.0,0.027344,0.136719,False
6,KO-specific,E-distance,9,4.646454,4.488725,0.000000,45.0,1.000000,1.000000,False
7,KO-specific,RMSE_LFC,9,0.410688,0.444962,0.555556,14.0,0.179688,0.539062,False


In [5]:
display_df = stats_df.copy()
display_df["p_value"] = display_df["p_value"].map(lambda p: f"{p:.3g}")
display_df["p_adj"] = display_df["p_adj"].map(lambda p: f"{p:.3g}")
display_df = display_df.set_index(["type", "metric"])[
    ["n_pairs", "mean_a", "mean_b", "cellina_base_win_rate", "statistic", "p_value", "p_adj", "significant"]
]
display_df

n_pairs    mean_a    mean_b  cellina_base_win_rate  \
type        metric                                                           
full        Pearson           9  0.583990  0.216109               0.777778   
            Precision         9  0.788889  0.584444               0.888889   
            E-distance        9  4.633645  4.454575               0.000000   
            RMSE_LFC          9  0.463146  0.546406               0.777778   
KO-specific Pearson           9  0.321283 -0.036876               0.666667   
            Precision         9  0.666667  0.464444               0.666667   
            E-distance        9  4.646454  4.488725               0.000000   
            RMSE_LFC          9  0.410688  0.444962               0.555556   

                        statistic  p_value   p_adj  significant  
type        metric                                               
full        Pearson          42.0  0.00977  0.0781        False  
            Precision        41.0   0.0137  0.0957        False  
            E-distance       45.0        1       1        False  
            RMSE_LFC          5.0   0.0195   0.117        False  
KO-specific Pearson          37.0   0.0488   0.195        False  
            Precision        39.0   0.0273   0.137        False  
            E-distance       45.0        1       1        False  
            RMSE_LFC         14.0     0.18   0.539        False